# Базовый пример: DistilBERT + 🤗 Transformers

В этом ноутбуке:
- загружаем предобученный DistilBERT,
- токенизируем текст,
- запускаем инференс через `pipeline`,
- финетюним модель на игрушечном датасете классификации.


In [1]:
!pip3 install -q transformers datasets accelerate torch

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## 2. Быстрый старт: `pipeline` для классификации

`pipeline` — самый простой вход в библиотеку 🤗 Transformers.  
Он:
- по имени задачи (`"text-classification"`, `"question-answering"` и т.п.) подбирает **подходящий тип модели** (`AutoModelForSequenceClassification` и т.д.),
- по имени чекпоинта с Hugging Face Hub скачивает **веса модели и токенизатор**,
- инкапсулирует все шаги: токенизация → прогон через модель → постобработка (softmax, argmax, маппинг в label-строки).

Это «high-level API»: удобно для быстрых экспериментов и демо, но под капотом те же базовые классы, которыми мы будем пользоваться дальше.

In [1]:
from transformers import pipeline

clf = pipeline(
    task="text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english"  # готовый sentiment-классификатор
)  # [web:97][web:103]

texts = [
    "This course on transformers is amazing!",
    "I hate debugging CUDA out-of-memory errors."
]

for t in texts:
    print(t, "->", clf(t))

Device set to use mps:0


This course on transformers is amazing! -> [{'label': 'POSITIVE', 'score': 0.9998784065246582}]
I hate debugging CUDA out-of-memory errors. -> [{'label': 'NEGATIVE', 'score': 0.9988393187522888}]


## 3. Низкоуровневый доступ: `AutoTokenizer` + `AutoModelForSequenceClassification`

Более гибкий и прозрачный уровень — явное использование базовых классов:

- `AutoTokenizer` — по имени модели скачивает и восстанавливает **точно тот же токенизатор**, на котором модель обучалась (тип токенизации, словарь, спецтокены).
- `AutoModelForSequenceClassification` — загружает предобученный **энкодер + классификационную голову** для задач вроде sentiment / токсичность / topic classification.
- Семейство `AutoModel*` подбирает правильный класс под конкретную архитектуру (BERT, RoBERTa, DistilBERT, GPT-2 и т.п.) по строке с именем модели.

Здесь мы:
1. Берём базовый `distilbert-base-uncased` (ещё не обученный на нашей задаче).
2. Определяем, что у нас будет `num_labels=2` (бинарная классификация).
3. Превращаем сырую строку в тензора (`input_ids`, `attention_mask`) через токенизатор и прогоняем через модель.

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL_NAME = "distilbert-base-uncased"  # базовая модель без финетюнинга [web:103]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2  # бинарная классификация
)

text = "Transformers make NLP for biology much easier."
encoded = tokenizer(text, return_tensors="pt")
with torch.no_grad():
    outputs = model(**encoded)
logits = outputs.logits
probs = torch.softmax(logits, dim=-1)
probs

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tensor([[0.4972, 0.5028]])

## 4. Игрушечный датасет для финетюнинга

Для работы с текстовыми датасетами удобно использовать библиотеку 🤗 `datasets`:

- `Dataset.from_dict` создаёт объект датасета из обычных Python-списков.
- Дальше мы будем применять к нему токенизацию батчами и приводить к формату, который ожидает `Trainer`.

Важно: Hugging Face экосистема старается сделать так, чтобы **данные, модель и обучение были максимально декларативны**:
- датасет знает про свои колонки (`"text"`, `"label"`),
- токенизатор знает, как превратить `"text"` в тензора,
- модель/Trainer знают, какие тензора им нужны (`input_ids`, `attention_mask`, `labels`).

In [3]:
from datasets import Dataset

# Простой датасет: 0 = негатив, 1 = позитив
train_texts = [
    "I am very happy with these results",
    "This model works great on our data",
    "The experiment completely failed",
    "I am disappointed with the performance",
]
train_labels = [1, 1, 0, 0]

train_ds = Dataset.from_dict({"text": train_texts, "label": train_labels})
train_ds

Dataset({
    features: ['text', 'label'],
    num_rows: 4
})

## 5. Токенизация датасета

На этом шаге мы:
- применяем один и тот же токенизатор к каждому примеру в датасете (`map(batched=True)`),
- переименовываем колонку `label` в `labels` — так её ожидает `Trainer`,
- указываем формát `set_format(type="torch", columns=[...])`, чтобы на выходе из датасета сразу получать PyTorch-тензора.

Результат: `tokenized_train` — объект, который можно без доп. обёрток скормить в `Trainer`.


In [4]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

tokenized_train = train_ds.map(tokenize_batch, batched=True)

# Hugging Face Trainer ожидает колонки input_ids, attention_mask, labels
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [5]:
tokenized_train

Dataset({
    features: ['text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 4
})

## 6. Финетюнинг с `Trainer`

`Trainer` — высокоуровневый обучающий цикл для большинства стандартных задач:

- Берёт на вход модель, датасеты (train/eval) и набор **гиперпараметров обучения** (`TrainingArguments`).
- Сам создаёт оптимизатор, шедулер learning rate, логи, чекпоинты и т.д.
- Вызывает `compute_metrics` на валидации, если она задана.

`TrainingArguments` задаёт:
- `output_dir` — куда складывать чекпоинты и логи,
- `learning_rate`, `num_train_epochs`, `per_device_train_batch_size`, `weight_decay` и т.п.,
- поведение логирования и сохранения (здесь мы оставили минимальный набор).

Этот блок — «учебный skeleton»: в реальных проектах меняется только:
- имя модели,
- гиперпараметры,
- датасеты.


In [6]:
!pip install -q --upgrade datasets
!pip install "numpy<2.0"


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [7]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./distilbert-demo",
    learning_rate=5e-5,
    per_device_train_batch_size=2,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=1
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    acc = (preds == labels).mean()
    return {"accuracy": acc}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_train,
    compute_metrics=compute_metrics,
)

trainer.train()

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,0.719200
2,0.829300
3,0.718400
4,0.613800
5,0.646900
6,0.620100


TrainOutput(global_step=6, training_loss=0.691283663113912, metrics={'train_runtime': 18.4151, 'train_samples_per_second': 0.652, 'train_steps_per_second': 0.326, 'total_flos': 198701097984.0, 'train_loss': 0.691283663113912, 'epoch': 3.0})

## 7. Проверяем модель после дообучения

In [9]:
test_texts = [
    "I am very satisfied with this model.",
    "The results are terrible and unstable."
]

encoded_batch = tokenizer(
    test_texts, 
    return_tensors="pt", 
    padding=True, 
    truncation=True, 
    max_length=64
)

encoded_batch = {k: v.to(model.device) for k, v in encoded_batch.items()}

with torch.no_grad():
    outputs = model(**encoded_batch)

probs = torch.softmax(outputs.logits, dim=-1)

for t, p in zip(test_texts, probs):
    print(t, "->", p)


I am very satisfied with this model. -> tensor([0.4670, 0.5330], device='mps:0')
The results are terrible and unstable. -> tensor([0.5243, 0.4757], device='mps:0')


## 8. Дообучение через LoRA (PEFT)

Полный финетюнинг обновляет **все** параметры модели — это дорого по памяти и времени, особенно для больших моделей.

Подход PEFT (Parameter-Efficient Fine-Tuning) и, в частности, LoRA:
- добавляет маленькие **адаптерные матрицы низкого ранга** к существующим весам (обычно к матрицам внимания),
- обучает только эти дополнительные параметры, а исходные веса модели **замораживает**,
- позволяет хранить для разных задач только небольшие «дельты» (LoRA-адаптеры), а базовая модель остаётся общей.

Ниже — пример, как обернуть наш DistilBERT в LoRA-адаптеры с помощью библиотеки `peft`.


In [11]:
!pip install peft -q
from peft import LoraConfig, get_peft_model

# Базовая модель из предыдущих шагов:
# model = AutoModelForSequenceClassification.from_pretrained(...)

lora_config = LoraConfig(
    r=8,                    # ранг низкоранговых матриц
    lora_alpha=16,          # масштабирующий коэффициент
    lora_dropout=0.1,       # dropout внутри LoRA
    bias="none",
    task_type="SEQ_CLS",    # тип задачи: sequence classification
)

lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 1.4 MB/s  0:00:00 eta 0:00:01

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


ValueError: Please specify `target_modules` or `target_parameters`in `peft_config`

Здесь:
- `get_peft_model` оборачивает исходную модель и добавляет LoRA-слои поверх выбранных матриц (по умолчанию — в attention).
- `print_trainable_parameters()` покажет, что обучаемых параметров стало **на порядки меньше**, чем при полном финетюнинге.

Важно: API модели (`forward(**batch)`) остаётся тем же — мы просто передаём в `Trainer` новую `lora_model`.


In [ ]:
lora_training_args = TrainingArguments(
    output_dir="./distilbert-lora-demo",
    learning_rate=5e-4,             
    per_device_train_batch_size=2,
    num_train_epochs=5,
    weight_decay=0.0,
    logging_steps=1
)

lora_trainer = Trainer(
    model=lora_model,
    args=lora_training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_train,
    compute_metrics=compute_metrics,
)

lora_trainer.train()

In [ ]:
test_texts = [
    "I am very satisfied with this model.",
    "The results are terrible and unstable."
]
encoded_batch = tokenizer(
    test_texts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=64
)
with torch.no_grad():
    outputs = lora_model(**encoded_batch)
probs = torch.softmax(outputs.logits, dim=-1)
for t, p in zip(test_texts, probs):
    print(t, "->", p)
